## <b>Data Preprocessing

### 1 Dataset Loading and Data Analysis

Load the OCTMNIST dataset at 64×64 resolution and inspect its structure, including image dimensions, class labels, and dataset sizes. Visualize representative images to understand the dataset.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from medmnist import INFO

dataset_path = Path("../data/octmnist_64.npz")
data = np.load(dataset_path)

# Dataset metadata
info = INFO["octmnist"]
class_names = list(info["label"].values())
class_ids = list(info["label"].keys())

print("===== OCTMNIST Dataset Summary =====")
print(f"Number of classes : {len(class_names)}")
print(f"Class names       : {', '.join(class_names)}")

total = 0

print("Available keys:", data.files)

for split in ["train", "val", "test"]:
    images = data[f"{split}_images"]
    labels = data[f"{split}_labels"].flatten()
    total += len(images)

    print(f"\n{split.upper()} SET")
    print(f"Number of images : {len(images):,}")
    print(f"Image shape      : {images.shape[1:]}")
    print(f"Class counts     :")

    counts = np.bincount(labels, minlength=len(class_names))
    for name, count in zip(class_names, counts):
        print(f"  {name:<8}: {count:,}")

print(f"\nTotal images     : {total:,}")

===== OCTMNIST Dataset Summary =====
Number of classes : 4
Class names       : choroidal neovascularization, diabetic macular edema, drusen, normal
Available keys: ['train_images', 'train_labels', 'val_images', 'val_labels', 'test_images', 'test_labels']

TRAIN SET
Number of images : 97,477
Image shape      : (64, 64)
Class counts     :
  choroidal neovascularization: 33,484
  diabetic macular edema: 10,213
  drusen  : 7,754
  normal  : 46,026

VAL SET
Number of images : 10,832
Image shape      : (64, 64)
Class counts     :
  choroidal neovascularization: 3,721
  diabetic macular edema: 1,135
  drusen  : 862
  normal  : 5,114

TEST SET
Number of images : 1,000
Image shape      : (64, 64)
Class counts     :
  choroidal neovascularization: 250
  diabetic macular edema: 250
  drusen  : 250
  normal  : 250

Total images     : 109,309


### 2 Image Resolution Downscaling

The OCTMNIST dataset was downloaded at 64×64 pixel resolution, satisfying the assignment's maximum image resolution requirement. Therefore, no additional downscaling was necessary.

### 3 Dataset Splitting

Combine the original dataset subsets and perform a stratified 70:15:15 split for training, validation, and testing, as required by the assignment. Verify the splitting ratios and class distributions.

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split

X_all = np.concatenate([
    data["train_images"],
    data["val_images"],
    data["test_images"]
], axis=0)

y_all = np.concatenate([
    data["train_labels"],
    data["val_labels"],
    data["test_labels"]
], axis=0).flatten()

print("All images:", X_all.shape)
print("All labels:", y_all.shape)


All images: (109309, 64, 64)
All labels: (109309,)


In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.30,
    random_state=42,
    stratify=y_all
)

print("Training:", X_train.shape)
print("Remaining:", X_temp.shape)

Training: (76516, 64, 64)
Remaining: (32793, 64, 64)


In [4]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (76516, 64, 64)
Validation: (16396, 64, 64)
Testing: (16397, 64, 64)


Verify the splitting ratios and class distributions.

In [8]:
for set, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    counts = np.bincount(labels.flatten(), minlength=len(class_ids))

    print(f"\n{set} Set: ({len(labels)/total*100:.2f}%)")

    for class_id in class_ids:
        name = info["label"][class_id]
        count = counts[int(class_id)]
        print(f"  {class_id} {name}: {count}")



Training Set: (70.00%)
  0 choroidal neovascularization: 26218
  1 diabetic macular edema: 8119
  2 drusen: 6206
  3 normal: 35973

Validation Set: (15.00%)
  0 choroidal neovascularization: 5618
  1 diabetic macular edema: 1740
  2 drusen: 1330
  3 normal: 7708

Testing Set: (15.00%)
  0 choroidal neovascularization: 5619
  1 diabetic macular edema: 1739
  2 drusen: 1330
  3 normal: 7709
